In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from pyproj import Transformer
from pathlib import Path
from osgeo import gdal, ogr
from math import floor
import numpy as np
import json

ModuleNotFoundError: No module named 'osgeo'

In [ ]:
SHAPEFILE = r'C:\Users\htla\OneDrive - Norges vassdrags- og energidirektorat\Datasett (GIS)\GPX tracks - Strava\clean_data\tracks_survey_respondents\SURVEY_GPX_DATA_A_REGIONS.shp'
RUN_DISTANCE = r'C:\Users\htla\OneDrive - Norges vassdrags- og energidirektorat\Datasett (GIS)\Flow-Py - Norge\alpha_23\runout_distance.tif'
PRA = r'C:\Users\htla\OneDrive - Norges vassdrags- og energidirektorat\Datasett (GIS)\Flow-Py - Norge\pra.tif'

In [ ]:
gdal.UseExceptions()

def extract_values_from_raster(src_filename, line_gdf):
    """
    Extract raster values at the locations of point geometries
    in a GeoDataFrame.

    Parameters
    ----------
    src_filename : str or pathlib.Path
        Path to raster file.

    line_gdf : geopandas.GeoDataFrame
        GeoDataFrame containing Point geometries.

    Returns
    -------
    np.ndarray
        Raster value for each point. Points outside the raster
        or on NoData cells are returned as np.nan.
    """

    # --------------------------------------------------------
    # Check raster path
    # --------------------------------------------------------

    src_filename = Path(src_filename)

    if not src_filename.exists():
        raise FileNotFoundError(
            f"Raster file not found:\n{src_filename}"
        )

    if not src_filename.is_file():
        raise ValueError(
            f"Raster path is not a file:\n{src_filename}"
        )

    # --------------------------------------------------------
    # Open raster
    # --------------------------------------------------------

    try:
        src_ds = gdal.Open(str(src_filename), gdal.GA_ReadOnly)

    except Exception as e:
        raise RuntimeError(
            f"GDAL could not open raster:\n{src_filename}\n\n"
            f"GDAL error:\n{e}"
        )

    if src_ds is None:
        raise RuntimeError(
            f"GDAL returned None when opening raster:\n{src_filename}"
        )

    # --------------------------------------------------------
    # Raster information
    # --------------------------------------------------------

    gt_forward = src_ds.GetGeoTransform()
    gt_reverse = gdal.InvGeoTransform(gt_forward)

    rb = src_ds.GetRasterBand(1)

    nodata = rb.GetNoDataValue()

    raster_width = src_ds.RasterXSize
    raster_height = src_ds.RasterYSize

    # --------------------------------------------------------
    # Check CRS
    # --------------------------------------------------------

    raster_wkt = src_ds.GetProjection()

    if not raster_wkt:
        raise ValueError(
            f"Raster has no CRS defined:\n{src_filename}"
        )

    if line_gdf.crs is None:
        raise ValueError(
            "The GeoDataFrame has no CRS defined."
        )

    raster_srs = osr.SpatialReference()
    raster_srs.ImportFromWkt(raster_wkt)

    raster_epsg = raster_srs.GetAuthorityCode(None)

    # --------------------------------------------------------
    # Reproject points if necessary
    # --------------------------------------------------------

    sample_gdf = line_gdf

    if raster_epsg is not None:

        raster_crs = f"EPSG:{raster_epsg}"

        if sample_gdf.crs != raster_crs:
            sample_gdf = sample_gdf.to_crs(raster_crs)

    else:
        # Fallback when the raster CRS does not have an EPSG code
        from pyproj import CRS

        raster_crs = CRS.from_wkt(raster_wkt)

        if sample_gdf.crs != raster_crs:
            sample_gdf = sample_gdf.to_crs(raster_crs)

    # --------------------------------------------------------
    # Extract raster values
    # --------------------------------------------------------

    values = []

    for geom in sample_gdf.geometry:

        if geom is None or geom.is_empty:
            values.append(np.nan)
            continue

        mx = geom.x
        my = geom.y

        # Convert map coordinates to raster pixel coordinates
        px, py = gdal.ApplyGeoTransform(
            gt_reverse,
            mx,
            my
        )

        px = floor(px)
        py = floor(py)

        # ----------------------------------------------------
        # Check whether point is inside raster
        # ----------------------------------------------------

        if (
            px < 0
            or py < 0
            or px >= raster_width
            or py >= raster_height
        ):
            values.append(np.nan)
            continue

        # ----------------------------------------------------
        # Read raster value
        # ----------------------------------------------------

        try:
            value_array = rb.ReadAsArray(
                px,
                py,
                1,
                1
            )

        except Exception:
            values.append(np.nan)
            continue

        if value_array is None:
            values.append(np.nan)
            continue

        value = value_array[0, 0]

        # ----------------------------------------------------
        # Handle NoData
        # ----------------------------------------------------

        if nodata is not None and np.isclose(value, nodata):
            values.append(np.nan)
            continue

        values.append(float(value))

    # Close raster
    src_ds = None

    return np.array(values, dtype=float)

In [ ]:
%%time

gdf = gpd.read_file(SHAPEFILE)

acc_pra = 0.75
acc_ro = 0.25
trip_pra = 0.1794
trip_ro = 0.8206
rr = (acc_ro/trip_ro)/(acc_pra/trip_pra)

data = pd.DataFrame()
rows = []

for i in range(0, 3, 1):
    print(i)
    row = {}
    track = gdf['geometry'][i]
    
    # Generate points every 10 m along line
    distance_delta = 10
    distances = np.arange(0, track.length, distance_delta)
    
    points = [track.interpolate(distance) for distance in distances]
    points.append(track.interpolate(track.length))
    
    # Create GeoDataFrame directly from points
    line_gdf = gpd.GeoDataFrame(geometry=points, crs=gdf.crs)
    line_gdf = line_gdf.reset_index(drop=True)
    
    # Extract ExpScale_Runout
    run = extract_values_from_raster(RUN_DISTANCE, line_gdf)
    
    run[np.where(run == 10000)] = 0
    lam = 0.01639453
    alpha=0.8153966
    run = np.exp(-(lam*run)**alpha)
    run[np.where(run == 1)] = 0
    # row['ExpScore_Runout_data'] = json.dumps(np.round(run, 4).tolist())
    row['ExpScore_Runout'] = np.nansum(run)*rr
    
    # Extract ExpScore_PRA
    pra = extract_values_from_raster(PRA, line_gdf)
    pra[np.where(pra == -128)] = 0
    pra = pra/100
    new_min = rr
    new_max = 1
    renormalized_pra = np.where(pra == 0, 0, new_min + (new_max - new_min) * pra)
    # row['ExpScore_PRA_data'] = json.dumps(np.round(renormalized_pra, 4).tolist())
    row['ExpScore_PRA'] = np.nansum(renormalized_pra)
    
    row['ExpScore'] = (row['ExpScore_Runout'])+row['ExpScore_PRA']
        
    rows.append(row)

# Convert all dictionaries to DataFrame
data = pd.DataFrame(rows)

In [ ]:
data

In [ ]:
acc_pra = 0.75
acc_ro = 0.25
trip_pra = 0.1794
trip_ro = 0.8206
rr = (acc_ro/trip_ro)/(acc_pra/trip_pra)

In [ ]:
data = pd.DataFrame()

In [ ]:
i = 50
track = gdf['geometry'][i]
row = gdf[['filename', 'datetime']].iloc[i:i+1]
date = row.iloc[0]['datetime']

In [ ]:
track

In [ ]:
# Generate points every 10 m along line
distance_delta = 10
distances = np.arange(0, track.length, distance_delta)

points = [track.interpolate(distance) for distance in distances]
points.append(track.interpolate(track.length))

# Create GeoDataFrame directly from points
line_gdf = gpd.GeoDataFrame(geometry=points, crs=gdf.crs)  # or whatever CRS your track uses
line_gdf = line_gdf.reset_index(drop=True)

In [ ]:
run = extract_values_from_raster(RUN_DISTANCE, line_gdf)
run[np.where(run == 10000)] = 0
lam = 0.01639453
alpha=0.8153966
run = np.exp(-(lam*run)**alpha)
run[np.where(run == 1)] = 0
row['ExpScore_Runout_data'] = json.dumps(np.round(run, 4).tolist())
row['ExpScore_Runout'] = run.sum()

In [ ]:
# Extract ExpScore_PRA
pra = extract_values_from_raster(ExpScore_PRA, line_gdf)
pra[np.where(pra == -128)] = 0
pra = pra/100
new_min = rr
new_max = 1
renormalized_pra = np.where(pra == 0, 0, new_min + (new_max - new_min) * pra)
row['ExpScore_PRA_data'] = json.dumps(np.round(renormalized_pra, 4).tolist())
row['ExpScore_PRA'] = renormalized_pra.sum()

In [ ]:
# Extract ELEVATION
ele = extract_values_from_raster(ELEVATION, line_gdf)
# pra[np.where(pra == 128)] = 0
row['elevation'] = json.dumps(np.round(ele, 4).tolist())
row['elevation_maks'] = ele.max()
row['elevation_min'] = ele.min()

# Extract SLOPE
slo = extract_values_from_raster(SLOPE, line_gdf)
# pra[np.where(pra == 128)] = 0
row['slope'] = json.dumps(np.round(slo, 4).tolist())

# Extract REGION
reg = extract_values_from_raster(REGION, line_gdf)
# pra[np.where(pra == 128)] = 0
mode_result = stats.mode(reg, nan_policy='omit', keepdims=False)
row['regionID'] = mode_result.mode

# Extract ASPECT
asp = extract_values_from_raster(ASPECT, line_gdf)
# pra[np.where(pra == 128)] = 0
row['aspect'] = json.dumps(np.round(asp, 4).tolist())
mode_result = stats.mode(asp, nan_policy='omit', keepdims=False)
row['aspect_mode'] = mode_result.mode
row['length'] = len(asp)*10

In [ ]:
# Filter forecast
region = int(row.iloc[0].regionID)
filtered_forecast = forecast[(forecast['Dato'] == date) & (forecast['Region-ID'] == region)]
filtered_forecast = filtered_forecast.fillna(np.nan)

In [ ]:
filtered_forecast

In [ ]:
row

In [ ]:
# Check if the DataFrame is empty
row['danger_level'] = filtered_forecast.iloc[0].Faregrad
row['SP1'] = filtered_forecast.iloc[0].SP1
row['SP1_size'] = filtered_forecast['Skredstørrelse SP1'].iloc[0]
row['SP1_distribution'] = filtered_forecast['Utbredelse SP1'].iloc[0]
row['SP1_sensitivity'] = filtered_forecast['Utløsbarhet SP1'].iloc[0]  

row['SP2'] = filtered_forecast.iloc[0].SP2
row['SP2_size'] = filtered_forecast['Skredstørrelse SP2'].iloc[0]
row['SP2_distribution'] = filtered_forecast['Utbredelse SP2'].iloc[0]
row['SP2_sensitivity'] = filtered_forecast['Utløsbarhet SP2'].iloc[0]

row['SP3'] = filtered_forecast.iloc[0].SP3
row['SP3_size'] = filtered_forecast['Skredstørrelse SP3'].iloc[0]
row['SP3_distribution'] = filtered_forecast['Utbredelse SP3'].iloc[0]
row['SP3_sensitivity'] = filtered_forecast['Utløsbarhet SP3'].iloc[0]

# Extract data from string
row.iloc[0].elevation
row.iloc[0].aspect
row.iloc[0].ExpScore_PRA_data
row.iloc[0].ExpScore_Runout_data

SP1_asp = []
SP1_min = []
SP1_max = []        
SP2_asp = []
SP2_min = []
SP2_max = []
SP3_asp = []
SP3_min = []
SP3_max = []

In [ ]:
row['SP1_asp'] = filtered_forecast.iloc[0, 8]
row['SP1_min'] = filtered_forecast.iloc[0, 9]
row['SP1_max'] = filtered_forecast.iloc[0, 10]
row['ExpScore_SP1_runout'], row['ExpScore_SP1_pra'], row['SP1_num_exposed_cells'], row['SP1_num_exposed_cells_runout'], row['SP1_num_exposed_cells_pra'] = get_exposed_sector(row.iloc[0].SP1_asp, row.iloc[0].SP1_min, row.iloc[0].SP1_max, row.iloc[0].elevation, row.iloc[0].aspect, row.iloc[0].ExpScore_Runout_data, row.iloc[0].ExpScore_PRA_data)

In [ ]:
SP_asp = row.iloc[0].SP1_asp
SP_min = row.iloc[0].SP1_min
SP_max = row.iloc[0].SP1_max
elevation = row.iloc[0].elevation
aspect = row.iloc[0].aspect
runout = row.iloc[0].ExpScore_Runout_data
pra = row.iloc[0].ExpScore_PRA_data

In [ ]:
SP_min, SP_max

In [ ]:
elevation[1500:2500]

In [ ]:
# Correct aspect dictionary according to specified format
aspect_dict = {'N': 1, 'NE': 2, 'E': 3, 'SE': 4, 'S': 5, 'SW': 6, 'W': 7, 'NW': 8}

# Filter aspects based on SP1_asp
valid_aspects = [aspect_dict[asp] for asp in SP_asp.split(', ')]

elevation = np.array(json.loads(elevation), dtype=float)
runout = np.array(json.loads(runout), dtype=float)
pra = np.array(json.loads(pra), dtype=float)
aspect = np.array(json.loads(aspect), dtype=float)

# Filter based on corrected aspect values and updated elevation criteria
valid_indices = (np.isin(aspect, valid_aspects) & (elevation >= SP_min) & (elevation <= SP_max))

# Count of valid values
num_valid_values = np.sum(valid_indices)

# Calculate sums for valid indices
count_runout = np.sum(runout[valid_indices] > 0)
count_pra = np.sum(pra[valid_indices] > 0)
sum_runout = np.sum(runout[valid_indices])
sum_pra = np.sum(pra[valid_indices])

In [ ]:
sum_pra

In [ ]:
valid_indices_asp = (np.isin(aspect, valid_aspects))

In [ ]:
valid_indices_asp[1523:2500]

In [ ]:
elevation[1523:2500]

In [ ]:
aspect

In [ ]:
row.ExpScore_SP1_runout

In [ ]:
row['ExpScore_SP2_runout'] = -1
row['ExpScore_SP2_pra'] = -1
row['SP2_num_exposed_cells'] = -1
row['SP2_asp'] = -1
row['SP2_min'] = -1
row['SP2_max'] = -1
row['SP2_num_exposed_cells_runout'] = -1
row['SP2_num_exposed_cells_pra'] = -1
row['SP1_2_num_exposed_cells_runout'] = -1
row['SP1_2_num_exposed_cells_pra'] = -1
row['SP1_2_num_exposed_cells'] = -1
row['ExpScore_SP1_2_runout'] = -1
row['ExpScore_SP1_2_pra'] = -1

In [ ]:
row['ExpScore_SP3_runout'] = -1
row['ExpScore_SP3_pra'] = -1
row['SP3_num_exposed_cells'] = -1
row['SP3_asp'] = -1
row['SP3_min'] = -1
row['SP3_max'] = -1
row['SP3_num_exposed_cells_runout'] = -1
row['SP3_num_exposed_cells_pra'] = -1
row['SP1_3_num_exposed_cells_runout'] = -1
row['SP1_3_num_exposed_cells_pra'] = -1
row['SP1_3_num_exposed_cells'] = -1
row['ExpScore_SP1_3_runout'] = -1
row['ExpScore_SP1_3_pra'] = -1

In [ ]:
        # print(row)
        columns_to_drop = ['elevation', 'slope', 'aspect', 'ExpScore_PRA_data', 'ExpScore_Runout_data']  # Replace with the names of the columns you want to drop
        row = row.drop(columns=columns_to_drop)
        
    data = data.append(row)

In [ ]:
# Correct aspect dictionary according to specified format
aspect_dict = {'N': 1, 'NE': 2, 'E': 3, 'SE': 4, 'S': 5, 'SW': 6, 'W': 7, 'NW': 8}

# Filter aspects based on SP1_asp
valid_aspects = [aspect_dict[asp] for asp in SP_asp.split(', ')]



# Filter based on corrected aspect values and updated elevation criteria
valid_indices = (np.isin(aspect, valid_aspects) & (elevation >= SP_min) & (elevation <= SP_max))

# Count of valid values
num_valid_values = np.sum(valid_indices)

# Calculate sums for valid indices
count_runout = np.sum(runout[valid_indices] > 0)
count_pra = np.sum(pra[valid_indices] > 0)
sum_runout = np.sum(runout[valid_indices])
sum_pra = np.sum(pra[valid_indices])

In [ ]:
print(type(row['SP1_min']), row['SP1_min'])
print(type(row['elevation']), row['elevation'])

In [ ]:
%%time

acc_pra = 0.75
acc_ro = 0.25
trip_pra = 0.1794
trip_ro = 0.8206
rr = (acc_ro/trip_ro)/(acc_pra/trip_pra)

data = pd.DataFrame()

for i in range(0, 10, 1):
    print(i)
    track = gdf['geometry'][i]
    row = gdf[['filename', 'datetime']].iloc[i:i+1]
    date = row.iloc[0]['datetime']
    
    # Generate points every 10 m along line
    distance_delta = 10
    distances = np.arange(0, track.length, distance_delta)
    
    points = [track.interpolate(distance) for distance in distances]
    points.append(track.interpolate(track.length))
    
    # Create GeoDataFrame directly from points
    line_gdf = gpd.GeoDataFrame(geometry=points, crs=gdf.crs)
    line_gdf = line_gdf.reset_index(drop=True)
    
    # Extract ExpScale_Runout
    run = extract_values_from_raster(RUN_DISTANCE, line_gdf)
    
    run[np.where(run == 10000)] = 0
    lam = 0.01639453
    alpha=0.8153966
    run = np.exp(-(lam*run)**alpha)
    run[np.where(run == 1)] = 0
    row['ExpScore_Runout_data'] = json.dumps(run.tolist())
    row['ExpScore_Runout'] = run.sum()
    
    # Extract ExpScore_PRA
    pra = extract_values_from_raster(ExpScore_PRA, line_gdf)
    pra[np.where(pra == -128)] = 0
    pra = pra/100
    new_min = rr
    new_max = 1
    renormalized_pra = np.where(pra == 0, 0, new_min + (new_max - new_min) * pra)
    row['ExpScore_PRA_data'] = json.dumps(renormalized_pra.tolist())
    row['ExpScore_PRA'] = renormalized_pra.sum()
    
    # row['ExpScore_PRA_data'] = [pra]
    # row['ExpScore_PRA'] = pra.sum()
    
    # row['ExpScore'] = (run.sum()*0.072857113)+pra.sum()
    # row['ExpScore'] = (run.sum()*0.072)+pra.sum()
    
    # Extract ELEVATION
    ele = extract_values_from_raster(ELEVATION, line_gdf)
    # pra[np.where(pra == 128)] = 0
    row['elevation'] = json.dumps(ele.tolist())
    row['elevation_maks'] = ele.max()
    row['elevation_min'] = ele.min()
    
    # Extract SLOPE
    slo = extract_values_from_raster(SLOPE, line_gdf)
    # pra[np.where(pra == 128)] = 0
    row['slope'] = json.dumps(slo.tolist())
    
    # Extract REGION
    reg = extract_values_from_raster(REGION, line_gdf)
    # pra[np.where(pra == 128)] = 0
    mode_result = stats.mode(reg, nan_policy='omit', keepdims=False)
    row['regionID'] = mode_result.mode
    
    # Extract ASPECT
    asp = extract_values_from_raster(ASPECT, line_gdf)
    # pra[np.where(pra == 128)] = 0
    row['aspect'] = json.dumps(asp.tolist())
    mode_result = stats.mode(asp, nan_policy='omit', keepdims=False)
    row['aspect_mode'] = mode_result.mode
    row['length'] = len(asp)*10
    
    # Filter forecast
    region = int(row.iloc[0].regionID)
    filtered_forecast = forecast[(forecast['Dato'] == date) & (forecast['Region-ID'] == region)]
    filtered_forecast = filtered_forecast.fillna(np.nan)
    
    # Check if the DataFrame is empty
    if filtered_forecast.empty:
        continue
        
    else:
        row['danger_level'] = filtered_forecast.iloc[0].Faregrad
        row['SP1'] = filtered_forecast.iloc[0].SP1
        row['SP1_size'] = filtered_forecast['Skredstørrelse SP1'].iloc[0]
        row['SP1_distribution'] = filtered_forecast['Utbredelse SP1'].iloc[0]
        row['SP1_sensitivity'] = filtered_forecast['Utløsbarhet SP1'].iloc[0]  

        row['SP2'] = filtered_forecast.iloc[0].SP2
        row['SP2_size'] = filtered_forecast['Skredstørrelse SP2'].iloc[0]
        row['SP2_distribution'] = filtered_forecast['Utbredelse SP2'].iloc[0]
        row['SP2_sensitivity'] = filtered_forecast['Utløsbarhet SP2'].iloc[0]

        row['SP3'] = filtered_forecast.iloc[0].SP3
        row['SP3_size'] = filtered_forecast['Skredstørrelse SP3'].iloc[0]
        row['SP3_distribution'] = filtered_forecast['Utbredelse SP3'].iloc[0]
        row['SP3_sensitivity'] = filtered_forecast['Utløsbarhet SP3'].iloc[0]
        
        # Extract data from string
        row.iloc[0].elevation
        row.iloc[0].aspect
        row.iloc[0].ExpScore_PRA_data
        row.iloc[0].ExpScore_Runout_data
        
        SP1_asp = []
        SP1_min = []
        SP1_max = []        
        SP2_asp = []
        SP2_min = []
        SP2_max = []
        SP3_asp = []
        SP3_min = []
        SP3_max = []

        # SP1
        if pd.isna(filtered_forecast.iloc[0, 8]) == True:
            row['ExpScore_SP1_runout'] = -1
            row['ExpScore_SP1_pra'] = -1
            row['SP1_num_exposed_cells'] = -1
            row['SP1_asp'] = -1
            row['SP1_min'] = -1
            row['SP1_max'] = -1
            row['SP1_num_exposed_cells_runout'] = -1
            row['SP1_num_exposed_cells_pra'] = -1
        else:
            row['SP1_asp'] = filtered_forecast.iloc[0, 8]
            row['SP1_min'] = filtered_forecast.iloc[0, 9]
            row['SP1_max'] = filtered_forecast.iloc[0, 10]
            row['ExpScore_SP1_runout'], row['ExpScore_SP1_pra'], row['SP1_num_exposed_cells'], row['SP1_num_exposed_cells_runout'], row['SP1_num_exposed_cells_pra'] = get_exposed_sector(row.iloc[0].SP1_asp, row.iloc[0].SP1_min, row.iloc[0].SP1_max, row.iloc[0].elevation, row.iloc[0].aspect, row.iloc[0].ExpScore_Runout_data, row.iloc[0].ExpScore_PRA_data)

        # SP2
        if pd.isna(filtered_forecast.iloc[0, 15]) == True:
            row['ExpScore_SP2_runout'] = -1
            row['ExpScore_SP2_pra'] = -1
            row['SP2_num_exposed_cells'] = -1
            row['SP2_asp'] = -1
            row['SP2_min'] = -1
            row['SP2_max'] = -1
            row['SP2_num_exposed_cells_runout'] = -1
            row['SP2_num_exposed_cells_pra'] = -1
            row['SP1_2_num_exposed_cells_runout'] = -1
            row['SP1_2_num_exposed_cells_pra'] = -1
            row['SP1_2_num_exposed_cells'] = -1
            row['ExpScore_SP1_2_runout'] = -1
            row['ExpScore_SP1_2_pra'] = -1
        else:
            row['SP2_asp'] = filtered_forecast.iloc[0, 15]
            row['SP2_min'] = filtered_forecast.iloc[0, 16]
            row['SP2_max'] = filtered_forecast.iloc[0, 17]
            row['ExpScore_SP2_runout'], row['ExpScore_SP2_pra'], row['SP2_num_exposed_cells'], row['SP2_num_exposed_cells_runout'], row['SP2_num_exposed_cells_pra'] = get_exposed_sector(row.iloc[0].SP2_asp, row.iloc[0].SP2_min, row.iloc[0].SP2_max, row.iloc[0].elevation, row.iloc[0].aspect, row.iloc[0].ExpScore_Runout_data, row.iloc[0].ExpScore_PRA_data)
            row['ExpScore_SP1_2_runout'], row['ExpScore_SP1_2_pra'], row['SP1_2_num_exposed_cells'], row['SP1_2_num_exposed_cells_runout'], row['SP1_2_num_exposed_cells_pra'] = get_exposed_sector_two(row.iloc[0].SP1_asp, row.iloc[0].SP1_min, row.iloc[0].SP1_max, row.iloc[0].SP2_asp, row.iloc[0].SP2_min, row.iloc[0].SP2_max, row.iloc[0].elevation, row.iloc[0].aspect, row.iloc[0].ExpScore_Runout_data, row.iloc[0].ExpScore_PRA_data)
        # SP3
        if pd.isna(filtered_forecast.iloc[0, 22]) == True:
            row['ExpScore_SP3_runout'] = -1
            row['ExpScore_SP3_pra'] = -1
            row['SP3_num_exposed_cells'] = -1
            row['SP3_asp'] = -1
            row['SP3_min'] = -1
            row['SP3_max'] = -1
            row['SP3_num_exposed_cells_runout'] = -1
            row['SP3_num_exposed_cells_pra'] = -1
            row['SP1_3_num_exposed_cells_runout'] = -1
            row['SP1_3_num_exposed_cells_pra'] = -1
            row['SP1_3_num_exposed_cells'] = -1
            row['ExpScore_SP1_3_runout'] = -1
            row['ExpScore_SP1_3_pra'] = -1
        else:
            row['SP3_asp'] = filtered_forecast.iloc[0, 22]
            row['SP3_min'] = filtered_forecast.iloc[0, 23]
            row['SP3_max'] = filtered_forecast.iloc[0, 24]
            row['ExpScore_SP3_runout'], row['ExpScore_SP3_pra'], row['SP3_num_exposed_cells'], row['SP3_num_exposed_cells_runout'], row['SP3_num_exposed_cells_pra'] = get_exposed_sector(row.iloc[0].SP3_asp, row.iloc[0].SP3_min, row.iloc[0].SP3_max, row.iloc[0].elevation, row.iloc[0].aspect, row.iloc[0].ExpScore_Runout_data, row.iloc[0].ExpScore_PRA_data)
            row['ExpScore_SP1_3_runout'], row['ExpScore_SP1_3_pra'], row['SP1_3_num_exposed_cells'], row['SP1_3_num_exposed_cells_runout'], row['SP1_3_num_exposed_cells_pra'] = get_exposed_sector_three(row.iloc[0].SP1_asp, row.iloc[0].SP1_min, row.iloc[0].SP1_max, row.iloc[0].SP2_asp, row.iloc[0].SP2_min, row.iloc[0].SP2_max, row.iloc[0].SP3_asp, row.iloc[0].SP3_min, row.iloc[0].SP3_max, row.iloc[0].elevation, row.iloc[0].aspect, row.iloc[0].ExpScore_Runout_data, row.iloc[0].ExpScore_PRA_data)
            
        # print(row)
        columns_to_drop = ['elevation', 'slope', 'aspect', 'ExpScore_PRA_data', 'ExpScore_Runout_data']  # Replace with the names of the columns you want to drop
        row = row.drop(columns=columns_to_drop)
        
    data = data.append(row)